# 2. Perceptron — experimento reproduzível

Este notebook implementa manualmente o perceptron para um conjunto separável (Exercício 1) e outro sobreposto (Exercício 2). O fluxo está dividido conforme os itens A–D do enunciado; as seis figuras e as métricas são geradas ao executar todas as células em ordem.

**Reprodutibilidade:** há um único gerador `rng = np.random.default_rng(42)`. A ordem das amostragens não deve ser alterada: dados do Exercício 1, inicialização `w0` do Exercício 1, dados do Exercício 2 e inicialização `w0` do Exercício 2. Os exemplos são mantidos na ordem gerada, com a classe 0 primeiro e sem embaralhamento.

In [ ]:
# Ambiente, fonte das funções e única semente aleatória do relatório.
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "perceptron.py").exists():
    NOTEBOOK_DIR = NOTEBOOK_DIR / "docs" / "exercises" / "perceptron" / "code"
FIGURES_DIR = NOTEBOOK_DIR.parent / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(NOTEBOOK_DIR))
from perceptron import accuracy, make_data, predict, train_perceptron

rng = np.random.default_rng(42)
plt.rcParams.update({"figure.figsize": (8, 6), "axes.grid": True, "grid.alpha": 0.2})

print(f"NumPy: {np.__version__}; pandas: {pd.__version__}")
print(f"Pasta das figuras: {FIGURES_DIR.resolve()}")

## 1–3. Exercício 1 — A: geração dos dados e Figura 1

Cada classe tem 1.000 amostras gaussianas bidimensionais. `make_data` empilha a classe 0 antes da classe 1 e não embaralha os dados. A covariância é $0.5I$. Logo após os dados, sorteia-se uma única inicialização não nula; a mesma cópia será usada nas duas taxas de aprendizagem.

In [ ]:
# Draw order 1 and 2: Exercise 1 data first, then its one shared initialization.
X1, y1 = make_data(
    rng,
    mean_class_0=[1.5, 1.5],
    mean_class_1=[5.0, 5.0],
    variance=0.5,
    samples_per_class=1000,
)
w0_ex1 = rng.normal(0.0, 0.01, size=2)
b0_ex1 = 0.0

assert X1.shape == (2000, 2) and y1.shape == (2000,)
assert np.all(y1[:1000] == 0) and np.all(y1[1000:] == 1)
print(f"X1: {X1.shape}; y1: {y1.shape}; w0 compartilhado: {w0_ex1}; b0: {b0_ex1}")

In [ ]:
# Figura 1: todas as observações, cores e legenda por classe.
fig, ax = plt.subplots()
ax.scatter(X1[y1 == 0, 0], X1[y1 == 0, 1], s=16, alpha=0.65, label="Classe 0")
ax.scatter(X1[y1 == 1, 0], X1[y1 == 1, 1], s=16, alpha=0.65, label="Classe 1")
ax.set(title="Figura 1 — dados separáveis do Exercício 1", xlabel="$x_1$", ylabel="$x_2$")
ax.legend(title="Classe")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "fig1.png", dpi=160, bbox_inches="tight")
plt.show()
plt.close(fig)

## 4–5. Exercício 1 — B: perceptron manual; C: duas taxas

A implementação única e comentada está em `perceptron.py` e é importada acima. Ela prevê 1 quando $w\cdot x+b\geq0$, atualiza com $e=y-\hat y$ para rótulos 0/1, percorre os dados na ordem fixa e para após uma época sem atualizações ou 100 épocas. A cada época registra a acurácia no conjunto completo e o total de atualizações.

In [ ]:
# As duas execuções compartilham dados e inicialização; só eta muda.
run_ex1_eta_001 = train_perceptron(
    X1, y1, w0_ex1.copy(), b0_ex1, learning_rate=0.01, max_epochs=100
)
run_ex1_eta_1 = train_perceptron(
    X1, y1, w0_ex1.copy(), b0_ex1, learning_rate=1.0, max_epochs=100
)

def describe_run(name, result):
    direction = result.weights / np.linalg.norm(result.weights)
    print(f"{name}: w={result.weights}, b={result.bias:.12g}")
    print(f"  ||w||={np.linalg.norm(result.weights):.12g}; w/||w||={direction}")
    print(f"  épocas={result.epochs}; acurácia final={result.accuracy_history[-1]:.6%}")
    print(f"  atualizações/época={result.updates_per_epoch}")

describe_run("Ex. 1, eta=0.01", run_ex1_eta_001)
describe_run("Ex. 1, eta=1.0", run_ex1_eta_1)


## 6. Exercício 1 — Figuras 2 e 3

A Figura 2 mostra as duas fronteiras finais para comparar as taxas; os pontos mal classificados são destacados em relação à execução principal ($\eta=0.01$). O traçado trata separadamente o caso de $w_2$ próximo de zero e mantém os limites dos eixos finitos. A Figura 3 mostra a acurácia por época da execução principal.

In [ ]:
def draw_boundary(ax, features, weights, bias, label, color, linestyle="-"):
    """Desenha a fronteira sem deixar a divisão por w[1] explodir os limites."""
    x_min, x_max = float(features[:, 0].min()), float(features[:, 0].max())
    y_min, y_max = float(features[:, 1].min()), float(features[:, 1].max())
    x_margin = 0.04 * (x_max - x_min)
    y_margin = 0.04 * (y_max - y_min)
    if abs(weights[1]) > 1e-12:
        x_line = np.linspace(x_min - x_margin, x_max + x_margin, 300)
        y_line = -(weights[0] * x_line + bias) / weights[1]
        ax.plot(x_line, y_line, color=color, linestyle=linestyle, linewidth=2.2, label=label)
    elif abs(weights[0]) > 1e-12:
        ax.axvline(-bias / weights[0], color=color, linestyle=linestyle, linewidth=2.2, label=label)
    else:
        raise ValueError("A fronteira não está definida para w = [0, 0].")
    ax.set_xlim(x_min - x_margin, x_max + x_margin)
    ax.set_ylim(y_min - y_margin, y_max + y_margin)

# Figura 2: classes, erros da corrida principal e fronteiras dos dois valores de eta.
fig, ax = plt.subplots(figsize=(9, 7))
ax.scatter(X1[y1 == 0, 0], X1[y1 == 0, 1], s=15, alpha=0.42, label="Classe 0")
ax.scatter(X1[y1 == 1, 0], X1[y1 == 1, 1], s=15, alpha=0.42, label="Classe 1")
wrong1 = predict(X1, run_ex1_eta_001.weights, run_ex1_eta_001.bias) != y1
ax.scatter(X1[wrong1, 0], X1[wrong1, 1], marker="x", color="black", s=50, linewidth=1.5, label="Mal classificados (eta=0.01)")
draw_boundary(ax, X1, run_ex1_eta_001.weights, run_ex1_eta_001.bias, "Fronteira eta=0.01", "crimson")
draw_boundary(ax, X1, run_ex1_eta_1.weights, run_ex1_eta_1.bias, "Fronteira eta=1.0", "darkgreen", "--")
ax.set(title="Figura 2 — fronteiras finais no Exercício 1", xlabel="$x_1$", ylabel="$x_2$")
ax.legend(loc="best", fontsize="small")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "fig2.png", dpi=160, bbox_inches="tight")
plt.show()
plt.close(fig)

In [ ]:
# Figura 3: acurácia do estado atual ao fim de cada época, corrida eta=0.01.
fig, ax = plt.subplots()
epoch_axis = np.arange(1, run_ex1_eta_001.epochs + 1)
ax.plot(epoch_axis, run_ex1_eta_001.accuracy_history, marker="o", label="Acurácia (eta=0.01)")
ax.set(title="Figura 3 — acurácia por época no Exercício 1", xlabel="Época", ylabel="Acurácia")
ax.set_ylim(0.0, 1.02)
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES_DIR / "fig3.png", dpi=160, bbox_inches="tight")
plt.show()
plt.close(fig)

## 7. Exercício 1 — D: convergência, efeito de $\eta$ e início zero

Uma época sem erros produz zero atualizações e satisfaz a parada. A lista numérica de atualizações por época, as acurácias e os parâmetros finais das duas execuções estão impressos acima; compare $w/\|w\|$, $b$ e $\|w\|$ com base nesses resultados, sem inferir diferenças de ângulo que os valores não sustentem.

**Prova para inicialização nula.** Seja $(w_t(1),b_t(1))$ a trajetória com taxa 1. Na base, ambas as trajetórias começam em zero. Suponha que $(w_t(\eta),b_t(\eta))=\eta(w_t(1),b_t(1))$. Como $\eta>0$, multiplicar $w\cdot x+b$ por $\eta$ preserva seu sinal e também o caso de igualdade com zero; portanto, as previsões e erros são iguais nas duas trajetórias. Então

$$w_{t+1}(\eta)=\eta w_t(1)+\eta e x=\eta(w_t(1)+e x)=\eta w_{t+1}(1),\qquad b_{t+1}(\eta)=\eta b_{t+1}(1).$$

Por indução, $w(\eta_2)=(\eta_2/\eta_1)w(\eta_1)$ e o bias escala pelo mesmo fator. Assim, a fronteira, a sequência de erros e a época de parada são idênticas para taxas positivas. Com uma inicialização não nula, a taxa muda o peso relativo entre inicialização e atualizações; equivalentemente, $(w,b)(\eta;w_0)=\eta(w,b)(1;w_0/\eta)$.

## 8. Exercício 2 — A: geração dos dados e Figura 4

Agora as médias são próximas e a variância é $1.5$ em cada dimensão. A geração preserva a mesma convenção (classe 0 primeiro, sem embaralhamento). A inicialização do Exercício 2 é sorteada depois desses dados, conforme a ordem reprodutível definida no início.

In [ ]:
# Draw order 3 and 4: Exercise 2 data followed by its new initialization.
X2, y2 = make_data(
    rng,
    mean_class_0=[3.0, 3.0],
    mean_class_1=[4.0, 4.0],
    variance=1.5,
    samples_per_class=1000,
)
w0_ex2 = rng.normal(0.0, 0.01, size=2)
b0_ex2 = 0.0

assert X2.shape == (2000, 2) and y2.shape == (2000,)
assert np.all(y2[:1000] == 0) and np.all(y2[1000:] == 1)
print(f"X2: {X2.shape}; y2: {y2.shape}; w0: {w0_ex2}; b0: {b0_ex2}")

In [ ]:
# Figura 4: as nuvens do conjunto não separável.
fig, ax = plt.subplots()
ax.scatter(X2[y2 == 0, 0], X2[y2 == 0, 1], s=16, alpha=0.58, label="Classe 0")
ax.scatter(X2[y2 == 1, 0], X2[y2 == 1, 1], s=16, alpha=0.58, label="Classe 1")
ax.set(title="Figura 4 — dados sobrepostos do Exercício 2", xlabel="$x_1$", ylabel="$x_2$")
ax.legend(title="Classe")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "fig4.png", dpi=160, bbox_inches="tight")
plt.show()
plt.close(fig)

## 9. Exercício 2 — B: treinamento com pocket

A função é exatamente a mesma. A única opção ativada é `use_pocket=True`: após cada atualização, a acurácia é recalculada sobre os 2.000 pontos; somente uma melhora estrita substitui o snapshot guardado. O histórico pocket registra o melhor valor ao fim de cada época.

In [ ]:
run_ex2 = train_perceptron(
    X2,
    y2,
    w0_ex2.copy(),
    b0_ex2,
    learning_rate=0.01,
    max_epochs=100,
    use_pocket=True,
)
final_ex2_accuracy = accuracy(X2, y2, run_ex2.weights, run_ex2.bias)
pocket_ex2_accuracy = accuracy(X2, y2, run_ex2.pocket_weights, run_ex2.pocket_bias)

describe_run("Ex. 2, pesos finais", run_ex2)
print(f"  acurácia final recalculada={final_ex2_accuracy:.6%}")
print(
    "Ex. 2, pocket: "
    f"w={run_ex2.pocket_weights}; b={run_ex2.pocket_bias:.12g}; "
    f"acurácia={pocket_ex2_accuracy:.6%}; época={run_ex2.pocket_epoch}"
)
print(f"  melhor acurácia por época={run_ex2.pocket_accuracy_history}")

## 10. Exercício 2 — C: Figuras 5 e 6

Na Figura 5, os dois conjuntos de erros têm marcadores diferentes, e as fronteiras final e pocket também são distinguidas. A Figura 6 compara a acurácia do estado atual ao melhor resultado conhecido até cada época.

In [ ]:
# A execução com pocket necessariamente preenche seus pesos e a curva correspondente.
assert run_ex2.pocket_weights is not None
assert run_ex2.pocket_bias is not None
assert run_ex2.pocket_accuracy_history is not None

# Figura 5: classes, erros das duas soluções e ambas as fronteiras.
fig, ax = plt.subplots(figsize=(9, 7))
ax.scatter(X2[y2 == 0, 0], X2[y2 == 0, 1], s=13, alpha=0.32, label="Classe 0")
ax.scatter(X2[y2 == 1, 0], X2[y2 == 1, 1], s=13, alpha=0.32, label="Classe 1")
wrong_final = predict(X2, run_ex2.weights, run_ex2.bias) != y2
wrong_pocket = predict(X2, run_ex2.pocket_weights, run_ex2.pocket_bias) != y2
ax.scatter(X2[wrong_final, 0], X2[wrong_final, 1], marker="x", color="black", s=34, label="Erros — final")
ax.scatter(X2[wrong_pocket, 0], X2[wrong_pocket, 1], marker="+", color="darkorange", s=42, label="Erros — pocket")
draw_boundary(ax, X2, run_ex2.weights, run_ex2.bias, "Fronteira final", "crimson")
draw_boundary(ax, X2, run_ex2.pocket_weights, run_ex2.pocket_bias, "Fronteira pocket", "navy", "--")
ax.set(title="Figura 5 — fronteiras final e pocket no Exercício 2", xlabel="$x_1$", ylabel="$x_2$")
ax.legend(loc="best", fontsize="small")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "fig5.png", dpi=160, bbox_inches="tight")
plt.show()
plt.close(fig)

# Figura 6: desempenho do estado corrente versus o melhor snapshot acumulado.
fig, ax = plt.subplots()
epoch_axis_ex2 = np.arange(1, run_ex2.epochs + 1)
ax.plot(epoch_axis_ex2, run_ex2.accuracy_history, marker=".", markersize=4, label="Acurácia atual")
ax.plot(epoch_axis_ex2, run_ex2.pocket_accuracy_history, linewidth=2, label="Melhor até então (pocket)")
ax.set(title="Figura 6 — acurácia por época no Exercício 2", xlabel="Época", ylabel="Acurácia")
ax.set_ylim(0.0, 1.02)
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES_DIR / "fig6.png", dpi=160, bbox_inches="tight")
plt.show()
plt.close(fig)

## 11. Exercício 2 — D: interpretação e comparação

A acurácia final mede o último estado do laço; a pocket preserva a melhor configuração encontrada. Em cada erro, o vetor peso muda em $\eta e x$, enquanto o bias muda em $\eta e$; como os vetores de entrada têm norma muito maior que 1, os giros da fronteira podem ser relevantes. O perceptron continua atualizando porque as observações não permitem uma fronteira linear perfeita; por isso, a última fronteira depende do último passo e pode cortar a nuvem de modo pouco útil. A localização e a diferença entre as acurácias devem ser lidas nas métricas e na Figura 5 geradas acima.

Na Figura 3, os erros acabam e a acurácia estabiliza no conjunto separável. Na Figura 6, a acurácia corrente pode oscilar e a curva pocket é não decrescente. O teorema de convergência do perceptron garante número finito de atualizações quando os dados são linearmente separáveis (e limitados); a sobreposição do Exercício 2 viola a hipótese de separabilidade. Mais épocas prolongam os erros sem garantir melhora; reduzir $\eta$ altera a escala/traçado dos passos, mas não cria um hiperplano separador e não garante que os erros desapareçam.

## 12. Results summary

Tabela final montada diretamente dos objetos produzidos nesta execução. Os valores não são redigitados manualmente; execute o notebook de cima a baixo para reproduzi-los.

In [ ]:
# Validações finais sem fazer qualquer sorteio adicional.
assert len(run_ex1_eta_001.updates_per_epoch) == run_ex1_eta_001.epochs
assert len(run_ex2.pocket_accuracy_history) == run_ex2.epochs
assert len(run_ex1_eta_001.accuracy_history) == run_ex1_eta_001.epochs
assert len(run_ex1_eta_1.accuracy_history) == run_ex1_eta_1.epochs
assert all((FIGURES_DIR / f"fig{i}.png").exists() for i in range(1, 7))

summary = pd.DataFrame(
    [
        (1, "Exercício 1: pesos finais w e bias b", f"w={run_ex1_eta_001.weights.tolist()}, b={run_ex1_eta_001.bias:.12g}"),
        (2, "Exercício 1: épocas até convergência", str(run_ex1_eta_001.epochs)),
        (3, "Exercício 1: acurácia final (eta=0.01)", f"{run_ex1_eta_001.accuracy_history[-1]:.6%}"),
        (4, "Exercício 1: épocas e acurácia final (eta=1.0)", f"{run_ex1_eta_1.epochs}; {run_ex1_eta_1.accuracy_history[-1]:.6%}"),
        (5, "Exercício 2: pesos finais w e bias b", f"w={run_ex2.weights.tolist()}, b={run_ex2.bias:.12g}"),
        (6, "Exercício 2: acurácia dos pesos finais", f"{final_ex2_accuracy:.6%}"),
        (7, "Exercício 2: acurácia dos pesos pocket", f"{pocket_ex2_accuracy:.6%}"),
        (8, "Exercício 2: época do melhor pocket", str(run_ex2.pocket_epoch)),
    ],
    columns=["#", "Métrica", "Valor"],
)
summary